# 06 — Class-Incremental Update + Replay Memory (Paper 3, Tahap T5)

**Tujuan:** tambahkan kelas serangan **baru** (held-out dari nb01) ke model multi-class
**tanpa catastrophic forgetting**, dengan **replay memory** sampel kelas lama. XGBoost
tree-ensemble tak bisa 'tambah kelas' native (num_class tetap) -> pendekatan jujur =
**retrain multi-class pada label-set diperluas** (known + kelas baru) memakai
(data kelas baru + buffer replay kelas lama). Konsisten documentation.md §7.

**Ablation inti:** forgetting **dengan vs tanpa** replay (bukti replay perlu).

**Masukan (dari nb01, auto-unduh S3 bila perlu):** `model_known_<DS>.json`,
`deploy_meta_mc_<DS>.json`. **Held-out** = 'kelas baru' (label dianggap sudah
diperoleh dari oracle nb05). **Keluaran -> S3 `evolusion/incremental/`.**

> HELDOUT & konvensi (merge_rare<200, split0.3, seed42) WAJIB identik nb01.
> JANGAN mengarang angka — semua dari eksekusi nyata di SageMaker.

## 1. Setup & konfigurasi (identik nb01)

In [ ]:
import importlib.util as u, sys, subprocess
for m in ('numpy','pandas','scikit-learn','xgboost','boto3'):
    if u.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, json, glob, datetime, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import f1_score, recall_score, matthews_corrcoef, balanced_accuracy_score, confusion_matrix
import xgboost as xgb
S3_BUCKET=os.environ.get('S3_BUCKET','ssh-detection-features-232032302717')
S3_PREFIX='evolusion'; REGION=os.environ.get('AWS_REGION','ap-southeast-1')
OUTDIR='incremental_out'; os.makedirs(OUTDIR,exist_ok=True)
CANON=['duration','fwd_pkts','bwd_pkts','fwd_bytes','bwd_bytes','fwd_mean','bwd_mean','src_load','dst_load']
SEED=42; MIN_CLASS=200
HELDOUT={'CIC':['Botnet','Infiltration'],'UNSW':['Worms','Shellcode','Backdoor']}  # == nb01
REPLAY_PER_CLASS=int(os.environ.get('REPLAY_PER_CLASS','2000'))  # buffer sampel per kelas lama
RESULTS={'generated':datetime.datetime.utcnow().isoformat()+'Z','features':CANON,'seed':SEED,
         'heldout':HELDOUT,'replay_per_class':REPLAY_PER_CLASS}
print('=== SEL 1 (config) SELESAI ===')

## 2. Muat data + pemetaan label (dipakai ulang dari nb01/nb24)

Isi `load_dataset(name)` memakai loader yang SAMA dengan nb01 (map_cic/map_unsw,
pilih file UNSW by jumlah record ~175k). Di sini disediakan kerangka + guard.

In [ ]:
# NOTE: salin fungsi map_cic/map_unsw & path dari nb01 SEL 2 agar identik.
# Placeholder loader: WAJIB diselaraskan dgn nb01 saat run (hindari skema label beda).
def merge_rare(df, min_n=MIN_CLASS):
    vc=df['cat'].value_counts(); rare=[c for c,n in vc.items() if n<min_n and c!='Benign']
    if rare: df=df.copy(); df['cat']=df['cat'].where(~df['cat'].isin(rare),'Other-rare')
    return df
def load_dataset(name):
    '''Kembalikan DataFrame dgn kolom CANON + kolom cat (kategori multi-class).
       IMPLEMENTASI: samakan dgn nb01 (map_cic/map_unsw). Return None bila data tak ada.'''
    print(f'[load_dataset] TODO: samakan dgn nb01 untuk {name} (jalankan di SageMaker).')
    return None
print('=== SEL 2 (loader placeholder) SELESAI ===')

## 3. Muat artefak model KNOWN dari nb01 (lokal / S3)

In [ ]:
def fetch(name_glob, s3key):
    h=sorted(glob.glob(name_glob))
    if h: return h[0]
    try:
        import boto3; s3=boto3.client('s3',region_name=REGION)
        local=os.path.join(OUTDIR, os.path.basename(s3key))
        s3.download_file(S3_BUCKET,s3key,local); print(' unduh S3:',s3key); return local
    except Exception as e: print(' gagal ambil',s3key,':',e); return None
def load_known(name):
    mp=fetch(f'**/deploy_meta_mc_{name}.json', f'{S3_PREFIX}/known_base/deploy_meta_mc_{name}.json')
    bp=fetch(f'**/model_known_{name}.json',    f'{S3_PREFIX}/known_base/model_known_{name}.json')
    if not (mp and bp): return None
    meta=json.load(open(mp)); booster=xgb.Booster(); booster.load_model(bp)
    return meta, booster
print('=== SEL 3 (fetch artefak known) SELESAI ===')

## 4. Inti T5: retrain class-incremental + replay, ablation forgetting

Alur per dataset:
1. Rekonstruksi split identik nb01 → dapatkan known-train/test + held-out (kelas baru).
2. **Replay buffer**: ambil ≤REPLAY_PER_CLASS sampel/kelas lama dari known-train.
3. **Dengan replay**: latih multi-class pada (replay lama + semua held-out-train).
4. **Tanpa replay (ablation)**: latih pada held-out-train saja (+ sedikit lama) → tunjukkan forgetting.
5. Ukur recall per-kelas pada test gabungan (lama+baru): forgetting = turunnya recall lama.

In [ ]:
def balanced_buffer(X,y,labels_old_idx,per_class,seed=SEED):
    rng=np.random.RandomState(seed); idx=[]
    for ci in labels_old_idx:
        ii=np.where(y==ci)[0]
        if len(ii)>per_class: ii=rng.choice(ii,per_class,replace=False)
        idx.append(ii)
    return np.concatenate(idx) if idx else np.array([],int)
def train_mc(X,y,num_class):
    m=xgb.XGBClassifier(objective='multi:softprob',num_class=num_class,max_depth=8,
        learning_rate=0.1,n_estimators=200,subsample=0.8,colsample_bytree=0.8,
        tree_method='hist',eval_metric='mlogloss',random_state=SEED,n_jobs=-1)
    m.fit(X,y); return m
def per_class_recall(y,yp,labels):
    out={}
    for ci,c in enumerate(labels):
        mask=(y==ci)
        out[c]=round(float(recall_score(mask,(yp==ci),zero_division=0)),4) if mask.any() else None
    return out
def run_incremental(name):
    df=load_dataset(name)
    if df is None: print(f'[{name}] data belum tersedia (jalankan di SageMaker)'); return None
    heldout=HELDOUT.get(name,[])
    df=merge_rare(df)
    df_known=df[~df['cat'].isin(heldout)].copy(); df_new=df[df['cat'].isin(heldout)].copy()
    # label-set DIPERLUAS: kelas lama + kelas baru
    all_labels=sorted(df_known['cat'].unique().tolist())+sorted(df_new['cat'].unique().tolist())
    le=LabelEncoder().fit(all_labels)
    def enc(d): return d[CANON].values, le.transform(d['cat'].values)
    Xk,yk=enc(df_known); Xn,yn=enc(df_new)
    Xk_tr,Xk_te,yk_tr,yk_te=train_test_split(Xk,yk,test_size=0.3,random_state=SEED,stratify=yk)
    Xn_tr,Xn_te,yn_tr,yn_te=train_test_split(Xn,yn,test_size=0.3,random_state=SEED,stratify=yn)
    sc=StandardScaler().fit(Xk_tr)                   # scaler known-train (no leakage dr baru)
    tr=lambda X: sc.transform(X)
    old_idx=sorted(np.unique(yk_tr).tolist()); nc=len(le.classes_)
    # (A) DENGAN replay
    bidx=balanced_buffer(Xk_tr,yk_tr,old_idx,REPLAY_PER_CLASS)
    XA=np.vstack([Xk_tr[bidx],Xn_tr]); yA=np.concatenate([yk_tr[bidx],yn_tr])
    mA=train_mc(tr(XA),yA,nc)
    # (B) TANPA replay (ablation) — hanya kelas baru + buffer SANGAT kecil (simulasi lupa)
    tiny=balanced_buffer(Xk_tr,yk_tr,old_idx,max(1,REPLAY_PER_CLASS//50))
    XB=np.vstack([Xk_tr[tiny],Xn_tr]); yB=np.concatenate([yk_tr[tiny],yn_tr])
    mB=train_mc(tr(XB),yB,nc)
    # Evaluasi pada test gabungan (lama+baru)
    Xte=np.vstack([Xk_te,Xn_te]); yte=np.concatenate([yk_te,yn_te]); Zte=tr(Xte)
    labels=list(le.classes_); new_set=set(le.transform(sorted(df_new['cat'].unique())))
    def summarize(m,tag):
        yp=np.asarray(m.predict(Zte)).ravel().astype(int)
        rc=per_class_recall(yte,yp,labels)
        old_r=[rc[c] for ci,c in enumerate(labels) if ci not in new_set and rc[c] is not None]
        new_r=[rc[c] for ci,c in enumerate(labels) if ci in new_set and rc[c] is not None]
        return {'tag':tag,'recall_per_class':rc,
                'mean_recall_old':round(float(np.mean(old_r)),4) if old_r else None,
                'mean_recall_new':round(float(np.mean(new_r)),4) if new_r else None,
                'mcc':round(float(matthews_corrcoef(yte,yp)),4)}
    rA=summarize(mA,'with_replay'); rB=summarize(mB,'without_replay')
    forget=None
    if rA['mean_recall_old'] is not None and rB['mean_recall_old'] is not None:
        forget=round(rA['mean_recall_old']-rB['mean_recall_old'],4)
    print(f'[{name}] OLD recall with={rA["mean_recall_old"]} vs without={rB["mean_recall_old"]} '
          f'| NEW recall with={rA["mean_recall_new"]} | replay_gain_old={forget}')
    return {'dataset':name,'labels':labels,'new_classes':sorted(df_new['cat'].unique().tolist()),
            'with_replay':rA,'without_replay':rB,'replay_gain_recall_old':forget}
print('=== SEL 4 (fungsi incremental+replay) SELESAI ===')

## 5. Jalankan + simpan + upload S3

In [ ]:
RESULTS['incremental']={}
for name in ['CIC','UNSW']:
    out=run_incremental(name)
    if out: RESULTS['incremental'][name]=out
jp=os.path.join(OUTDIR,'incremental_results.json'); json.dump(RESULTS,open(jp,'w'),indent=2)
print('tersimpan',jp)
try:
    import boto3; s3=boto3.client('s3',region_name=REGION); up=0
    for fn in sorted(os.listdir(OUTDIR)):
        if fn.endswith(('.json','.csv','.png')):
            s3.upload_file(os.path.join(OUTDIR,fn),S3_BUCKET,f'{S3_PREFIX}/incremental/{fn}'); up+=1
    print(f'upload {up} -> s3://{S3_BUCKET}/{S3_PREFIX}/incremental/')
except Exception as e: print('upload gagal:',e)
print('=== SEL 5 (simpan+upload) SELESAI ===')
print('SELESAI T5. replay_gain_recall_old > 0 = replay mencegah forgetting (bukti ablation).')

## 6. Catatan metodologi (untuk naskah)

- **Kejujuran:** XGBoost di-*retrain* dgn label-set diperluas (bukan incremental native);
  dilaporkan apa adanya sebagai strategi class-incremental utk tree-ensemble.
- **Ablation forgetting:** `with_replay` vs `without_replay` → `replay_gain_recall_old`
  positif membuktikan replay perlu. Model hasil = **kandidat**; promosi diputuskan nb07
  (guardrail: recall lama tak turun & baru naik).
- **Alur:** nb01 (known) → nb02/03 (unknown→cluster) → nb05 (label) → **nb06 (incremental)** → nb07 (guardrail).